# 03 — Credit Analysis

Calculate credit metrics and rank the five retailers.


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path


In [2]:
PROJECT_ROOT = Path("..")
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

financials_path = PROCESSED_DIR / "financials.csv"

if financials_path.exists():
    df = pd.read_csv(financials_path)
    display(df.head())
else:
    print("financials.csv has not been created yet.")


,company,fiscal_year,revenue,operating_income,net_income,cash,current_assets,current_liabilities,total_assets,total_debt,equity,operating_cash_flow,capex,interest_expense,depreciation_amortization
0,Costco,2021,195929000000,6708000000,5007000000,11258000000,29505000000,29441000000,59268000000,7.531000e+09,17564000000,8958000000,3588000000,171000000.0,1781000000
1,Costco,2022,226954000000,7793000000,5844000000,10203000000,32696000000,31998000000,64166000000,6.590000e+09,20642000000,7392000000,3891000000,158000000.0,1900000000
2,Costco,2023,242290000000,8114000000,6292000000,13700000000,35879000000,33583000000,68994000000,6.484000e+09,25058000000,11068000000,4323000000,160000000.0,2077000000
3,Costco,2024,254453000000,9285000000,7367000000,9906000000,34246000000,35464000000,69831000000,5.919000e+09,23622000000,11339000000,4710000000,169000000.0,2237000000
4,Costco,2025,275235000000,10383000000,8099000000,14161000000,38380000000,37108000000,77099000000,5.805000e+09,29164000000,13335000000,5498000000,154000000.0,2426000000


## Core credit metrics

- Revenue Growth
- Operating Margin
- Free Cash Flow
- Current Ratio
- Debt-to-Equity
- Interest Coverage
- Debt / EBITDA
- Operating Cash Flow / Debt


In [3]:
def calculate_credit_metrics(df):
    result = df.copy()

    result = result.sort_values(["company", "fiscal_year"])

    result["revenue_growth"] = (
        result.groupby("company")["revenue"].pct_change()
    )

    result["operating_margin"] = (
        result["operating_income"] / result["revenue"]
    )

    result["free_cash_flow"] = (
        result["operating_cash_flow"] - result["capex"]
    )

    result["current_ratio"] = (
        result["current_assets"] / result["current_liabilities"]
    )

    result["debt_to_equity"] = (
        result["total_debt"] / result["equity"]
    )

    result["interest_coverage"] = (
        result["operating_income"] / result["interest_expense"]
    )

    result["ebitda"] = (
        result["operating_income"] + result["depreciation_amortization"]
    )

    result["debt_to_ebitda"] = (
        result["total_debt"] / result["ebitda"]
    )

    result["ocf_to_debt"] = (
        result["operating_cash_flow"] / result["total_debt"]
    )

    return result


In [4]:
# ============================================================
# STEP 1 — LOAD CLEAN DATA INTO DUCKDB
# ============================================================

import pandas as pd
import duckdb
from pathlib import Path

PROJECT_ROOT = Path("..")
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

csv_path = PROCESSED_DIR / "financials.csv"
db_path = PROCESSED_DIR / "credit_analysis.duckdb"

# Connect to DuckDB
con = duckdb.connect(str(db_path))

# Load cleaned CSV
financials = pd.read_csv(csv_path)

print("CSV rows:", len(financials))
print("CSV columns:", len(financials.columns))

financials.head()

CSV rows: 25
CSV columns: 15


,company,fiscal_year,revenue,operating_income,net_income,cash,current_assets,current_liabilities,total_assets,total_debt,equity,operating_cash_flow,capex,interest_expense,depreciation_amortization
0,Costco,2021,195929000000,6708000000,5007000000,11258000000,29505000000,29441000000,59268000000,7.531000e+09,17564000000,8958000000,3588000000,171000000.0,1781000000
1,Costco,2022,226954000000,7793000000,5844000000,10203000000,32696000000,31998000000,64166000000,6.590000e+09,20642000000,7392000000,3891000000,158000000.0,1900000000
2,Costco,2023,242290000000,8114000000,6292000000,13700000000,35879000000,33583000000,68994000000,6.484000e+09,25058000000,11068000000,4323000000,160000000.0,2077000000
3,Costco,2024,254453000000,9285000000,7367000000,9906000000,34246000000,35464000000,69831000000,5.919000e+09,23622000000,11339000000,4710000000,169000000.0,2237000000
4,Costco,2025,275235000000,10383000000,8099000000,14161000000,38380000000,37108000000,77099000000,5.805000e+09,29164000000,13335000000,5498000000,154000000.0,2426000000


In [5]:
# ============================================================
# STEP 2 — CREATE FINANCIALS TABLE
# ============================================================

con.execute("""
DROP TABLE IF EXISTS financials;
""")

con.execute("""
CREATE TABLE financials AS
SELECT *
FROM financials
""")

result = con.execute("""
SELECT
    company,
    fiscal_year,
    revenue,
    operating_income,
    total_debt,
    operating_cash_flow
FROM financials
ORDER BY company, fiscal_year
""").df()

result

,company,fiscal_year,revenue,operating_income,total_debt,operating_cash_flow
0,Costco,2021,195929000000,6708000000,7.531000e+09,8958000000
1,Costco,2022,226954000000,7793000000,6.590000e+09,7392000000
2,Costco,2023,242290000000,8114000000,6.484000e+09,11068000000
3,Costco,2024,254453000000,9285000000,5.919000e+09,11339000000
4,Costco,2025,275235000000,10383000000,5.805000e+09,13335000000
5,Home Depot,2022,151157000000,23040000000,3.640000e+10,16571000000
6,Home Depot,2023,157403000000,24039000000,4.115000e+10,14615000000
7,Home Depot,2024,152669000000,21689000000,4.215000e+10,21172000000
8,Home Depot,2025,159514000000,21526000000,5.136500e+10,19810000000
9,Home Depot,2026,164683000000,20890000000,4.939700e+10,16325000000


In [6]:
# ============================================================
# STEP 3 — CREATE CREDIT METRICS TABLE
# ============================================================

con.execute("""
DROP TABLE IF EXISTS credit_metrics;
""")

con.execute("""
CREATE TABLE credit_metrics AS

WITH calculations AS (

    SELECT
        company,
        fiscal_year,

        revenue,
        operating_income,
        net_income,
        cash,
        current_assets,
        current_liabilities,
        total_assets,
        total_debt,
        equity,
        operating_cash_flow,
        capex,
        interest_expense,
        depreciation_amortization,

        LAG(revenue) OVER (
            PARTITION BY company
            ORDER BY fiscal_year
        ) AS previous_revenue

    FROM financials
)

SELECT
    *,

    (revenue - previous_revenue)
        / NULLIF(previous_revenue, 0)
        AS revenue_growth,

    operating_income
        / NULLIF(revenue, 0)
        AS operating_margin,

    operating_cash_flow - capex
        AS free_cash_flow,

    current_assets
        / NULLIF(current_liabilities, 0)
        AS current_ratio,

    total_debt
        / NULLIF(equity, 0)
        AS debt_to_equity,

    operating_income
        / NULLIF(interest_expense, 0)
        AS interest_coverage,

    operating_income + depreciation_amortization
        AS ebitda,

    total_debt
        / NULLIF(
            operating_income + depreciation_amortization,
            0
        )
        AS debt_to_ebitda,

    operating_cash_flow
        / NULLIF(total_debt, 0)
        AS ocf_to_debt

FROM calculations;
""")

In [7]:
metrics = con.execute("""
SELECT *
FROM credit_metrics
ORDER BY company, fiscal_year
""").df()

print("Rows:", len(metrics))
print("Columns:", len(metrics.columns))

metrics.head(10)

Rows: 25
Columns: 25


,company,fiscal_year,revenue,operating_income,net_income,cash,current_assets,current_liabilities,total_assets,total_debt,...,previous_revenue,revenue_growth,operating_margin,free_cash_flow,current_ratio,debt_to_equity,interest_coverage,ebitda,debt_to_ebitda,ocf_to_debt
0,Costco,2021,195929000000,6708000000,5007000000,11258000000,29505000000,29441000000,59268000000,7.531000e+09,...,<NA>,NaN,0.034237,5370000000,1.002174,0.428775,39.228070,8489000000,0.887148,1.189483
1,Costco,2022,226954000000,7793000000,5844000000,10203000000,32696000000,31998000000,64166000000,6.590000e+09,...,195929000000,0.158348,0.034337,3501000000,1.021814,0.319252,49.322785,9693000000,0.679872,1.121700
2,Costco,2023,242290000000,8114000000,6292000000,13700000000,35879000000,33583000000,68994000000,6.484000e+09,...,226954000000,0.067573,0.033489,6745000000,1.068368,0.258760,50.712500,10191000000,0.636248,1.706971
3,Costco,2024,254453000000,9285000000,7367000000,9906000000,34246000000,35464000000,69831000000,5.919000e+09,...,242290000000,0.050200,0.036490,6629000000,0.965655,0.250572,54.940828,11522000000,0.513713,1.915695
4,Costco,2025,275235000000,10383000000,8099000000,14161000000,38380000000,37108000000,77099000000,5.805000e+09,...,254453000000,0.081673,0.037724,7837000000,1.034278,0.199047,67.422078,12809000000,0.453197,2.297158
5,Home Depot,2022,151157000000,23040000000,16433000000,2343000000,29055000000,28693000000,71876000000,3.640000e+10,...,<NA>,NaN,0.152424,14005000000,1.012616,-21.462264,17.104677,25426000000,1.431605,0.455247
6,Home Depot,2023,157403000000,24039000000,17105000000,2757000000,32471000000,23110000000,76445000000,4.115000e+10,...,151157000000,0.041321,0.152723,11496000000,1.405063,26.344430,14.866419,26494000000,1.553182,0.355164
7,Home Depot,2024,152669000000,21689000000,15143000000,3760000000,29775000000,22015000000,76530000000,4.215000e+10,...,157403000000,-0.030076,0.142066,17946000000,1.352487,40.373563,11.162635,24362000000,1.730154,0.502301
8,Home Depot,2025,159514000000,21526000000,14806000000,1659000000,31683000000,28661000000,96119000000,5.136500e+10,...,152669000000,0.044836,0.134947,16325000000,1.105439,7.735693,9.274451,24560000000,2.091409,0.385671
9,Home Depot,2026,164683000000,20890000000,14156000000,1389000000,34391000000,32424000000,105095000000,4.939700e+10,...,159514000000,0.032405,0.126850,12646000000,1.060665,3.855225,8.660862,24163000000,2.044324,0.330486


In [8]:
# ============================================================
# LATEST-YEAR CREDIT COMPARISON
# ============================================================

latest_metrics = con.execute("""
WITH ranked AS (
    SELECT *,
           ROW_NUMBER() OVER (
               PARTITION BY company
               ORDER BY fiscal_year DESC
           ) AS rn
    FROM credit_metrics
)

SELECT
    company,
    fiscal_year,
    revenue_growth,
    operating_margin,
    free_cash_flow,
    current_ratio,
    debt_to_equity,
    interest_coverage,
    ebitda,
    debt_to_ebitda,
    ocf_to_debt
FROM ranked
WHERE rn = 1
ORDER BY company;
""").df()

latest_metrics

,company,fiscal_year,revenue_growth,operating_margin,free_cash_flow,current_ratio,debt_to_equity,interest_coverage,ebitda,debt_to_ebitda,ocf_to_debt
0,Costco,2025,0.081673,0.037724,7837000000,1.034278,0.199047,67.422078,12809000000,0.453197,2.297158
1,Home Depot,2026,0.032405,0.126850,12646000000,1.060665,3.855225,8.660862,24163000000,2.044324,0.330486
2,Lowes,2026,0.031216,0.117667,7651000000,1.076658,-4.015226,6.827841,12094000000,3.292459,0.247721
3,Target,2026,-0.016760,0.048836,2835000000,0.942299,0.890690,11.498876,7734000000,1.861650,0.455758
4,Walmart,2026,0.047255,0.042220,14923000000,0.789753,0.383127,12.866695,44028000000,0.866857,1.089058


In [9]:
# ============================================================
# CREDIT ANALYSIS VIEW
# HANDLE NEGATIVE EQUITY CORRECTLY
# ============================================================

credit_view = con.execute("""
WITH ranked AS (

    SELECT *,
        ROW_NUMBER() OVER (
            PARTITION BY company
            ORDER BY fiscal_year DESC
        ) AS rn

    FROM credit_metrics
)

SELECT

    company,
    fiscal_year,

    revenue_growth,
    operating_margin,
    free_cash_flow,
    current_ratio,

    equity,

    CASE
        WHEN equity <= 0 THEN NULL
        ELSE debt_to_equity
    END AS debt_to_equity,

    CASE
        WHEN equity <= 0 THEN 1
        ELSE 0
    END AS negative_equity_flag,

    interest_coverage,
    ebitda,
    debt_to_ebitda,
    ocf_to_debt

FROM ranked

WHERE rn = 1

ORDER BY company;
""").df()


# ============================================================
# FORMAT FOR EASY READING
# ============================================================

display_view = credit_view.copy()

display_view["revenue_growth"] = (
    display_view["revenue_growth"] * 100
).round(2)

display_view["operating_margin"] = (
    display_view["operating_margin"] * 100
).round(2)

display_view["free_cash_flow"] = (
    display_view["free_cash_flow"] / 1_000_000_000
).round(2)

display_view["equity"] = (
    display_view["equity"] / 1_000_000_000
).round(2)

display_view["current_ratio"] = (
    display_view["current_ratio"]
).round(2)

display_view["debt_to_equity"] = (
    display_view["debt_to_equity"]
).round(2)

display_view["interest_coverage"] = (
    display_view["interest_coverage"]
).round(2)

display_view["ebitda"] = (
    display_view["ebitda"] / 1_000_000_000
).round(2)

display_view["debt_to_ebitda"] = (
    display_view["debt_to_ebitda"]
).round(2)

display_view["ocf_to_debt"] = (
    display_view["ocf_to_debt"]
).round(2)


display_view = display_view.rename(columns={
    "revenue_growth": "revenue_growth_pct",
    "operating_margin": "operating_margin_pct",
    "free_cash_flow": "fcf_billions",
    "equity": "equity_billions"
})

display_view

,company,fiscal_year,revenue_growth_pct,operating_margin_pct,fcf_billions,current_ratio,equity_billions,debt_to_equity,negative_equity_flag,interest_coverage,ebitda,debt_to_ebitda,ocf_to_debt
0,Costco,2025,8.17,3.77,7.84,1.03,29.16,0.20,0,67.42,12.81,0.45,2.30
1,Home Depot,2026,3.24,12.68,12.65,1.06,12.81,3.86,0,8.66,24.16,2.04,0.33
2,Lowes,2026,3.12,11.77,7.65,1.08,-9.92,NaN,1,6.83,12.09,3.29,0.25
3,Target,2026,-1.68,4.88,2.84,0.94,16.16,0.89,0,11.50,7.73,1.86,0.46
4,Walmart,2026,4.73,4.22,14.92,0.79,99.62,0.38,0,12.87,44.03,0.87,1.09


In [10]:
# ============================================================
# PRELIMINARY CREDIT SCORING MODEL
# Score each company from 1 to 5
# Final stress-test score will be added later
# ============================================================

con.execute("""
DROP TABLE IF EXISTS preliminary_credit_scores;
""")

con.execute("""
CREATE TABLE preliminary_credit_scores AS

WITH historical AS (

    SELECT
        company,

        AVG(operating_cash_flow) AS avg_ocf,

        STDDEV_SAMP(operating_cash_flow)
            / NULLIF(AVG(operating_cash_flow), 0)
            AS ocf_cv,

        SUM(
            CASE
                WHEN free_cash_flow > 0 THEN 1
                ELSE 0
            END
        ) AS positive_fcf_years

    FROM credit_metrics

    GROUP BY company
),

latest AS (

    SELECT *

    FROM (

        SELECT
            *,
            ROW_NUMBER() OVER (
                PARTITION BY company
                ORDER BY fiscal_year DESC
            ) AS rn

        FROM credit_metrics
    )

    WHERE rn = 1
),

base AS (

    SELECT
        l.company,
        l.fiscal_year,

        l.revenue_growth,
        l.operating_margin,
        l.free_cash_flow,
        l.current_ratio,
        l.equity,
        l.total_debt,
        l.interest_coverage,
        l.debt_to_ebitda,
        l.ocf_to_debt,

        l.free_cash_flow
            / NULLIF(l.total_debt, 0)
            AS fcf_to_debt,

        h.ocf_cv,
        h.positive_fcf_years

    FROM latest l

    LEFT JOIN historical h
        ON l.company = h.company
),

scores AS (

    SELECT
        *,

        -- ==========================================
        -- LEVERAGE SCORE
        -- Lower Debt / EBITDA is better
        -- Negative equity receives strongest penalty
        -- ==========================================

        CASE
            WHEN equity <= 0 THEN 1
            WHEN debt_to_ebitda <= 1.0 THEN 5
            WHEN debt_to_ebitda <= 2.0 THEN 4
            WHEN debt_to_ebitda <= 3.0 THEN 3
            WHEN debt_to_ebitda <= 4.0 THEN 2
            ELSE 1
        END AS leverage_score,


        -- ==========================================
        -- INTEREST COVERAGE SCORE
        -- Higher coverage is better
        -- ==========================================

        CASE
            WHEN interest_coverage >= 10 THEN 5
            WHEN interest_coverage >= 6 THEN 4
            WHEN interest_coverage >= 3 THEN 3
            WHEN interest_coverage >= 1.5 THEN 2
            ELSE 1
        END AS coverage_score,


        -- ==========================================
        -- LIQUIDITY SCORE
        -- Retail-adjusted current-ratio thresholds
        -- ==========================================

        CASE
            WHEN current_ratio >= 1.20 THEN 5
            WHEN current_ratio >= 1.00 THEN 4
            WHEN current_ratio >= 0.85 THEN 3
            WHEN current_ratio >= 0.70 THEN 2
            ELSE 1
        END AS liquidity_score,


        -- ==========================================
        -- FREE CASH FLOW SCORE
        -- FCF relative to debt
        -- ==========================================

        CASE
            WHEN (
                free_cash_flow / NULLIF(total_debt, 0)
            ) >= 0.50 THEN 5

            WHEN (
                free_cash_flow / NULLIF(total_debt, 0)
            ) >= 0.30 THEN 4

            WHEN (
                free_cash_flow / NULLIF(total_debt, 0)
            ) >= 0.20 THEN 3

            WHEN (
                free_cash_flow / NULLIF(total_debt, 0)
            ) >= 0.10 THEN 2

            ELSE 1
        END AS fcf_score,


        -- ==========================================
        -- CASH-FLOW STABILITY SCORE
        -- Lower coefficient of variation is better
        -- ==========================================

        CASE
            WHEN ocf_cv <= 0.10 THEN 5
            WHEN ocf_cv <= 0.20 THEN 4
            WHEN ocf_cv <= 0.30 THEN 3
            WHEN ocf_cv <= 0.40 THEN 2
            ELSE 1
        END AS cash_flow_stability_score,


        -- ==========================================
        -- PROFITABILITY SCORE
        -- Operating margin
        -- ==========================================

        CASE
            WHEN operating_margin >= 0.12 THEN 5
            WHEN operating_margin >= 0.08 THEN 4
            WHEN operating_margin >= 0.05 THEN 3
            WHEN operating_margin >= 0.03 THEN 2
            ELSE 1
        END AS profitability_score

    FROM base
)

SELECT
    *,

    ROUND(
          leverage_score            * 0.25
        + coverage_score            * 0.20
        + liquidity_score           * 0.10
        + fcf_score                 * 0.20
        + cash_flow_stability_score * 0.15
        + profitability_score       * 0.10
    , 2) AS pre_stress_score

FROM scores;
""")


# ============================================================
# DISPLAY RANKING
# ============================================================

scorecard = con.execute("""
SELECT
    company,

    leverage_score,
    coverage_score,
    liquidity_score,
    fcf_score,
    cash_flow_stability_score,
    profitability_score,

    ROUND(fcf_to_debt, 2) AS fcf_to_debt,
    ROUND(ocf_cv, 2) AS ocf_cv,

    pre_stress_score

FROM preliminary_credit_scores

ORDER BY pre_stress_score DESC;
""").df()

scorecard

,company,leverage_score,coverage_score,liquidity_score,fcf_score,cash_flow_stability_score,profitability_score,fcf_to_debt,ocf_cv,pre_stress_score
0,Costco,5,5,4,5,3,2,1.35,0.22,4.30
1,Walmart,5,5,2,4,3,2,0.39,0.21,3.90
2,Home Depot,3,4,4,3,4,5,0.26,0.15,3.65
3,Target,4,5,3,2,3,2,0.20,0.27,3.35
4,Lowes,1,4,4,2,5,4,0.19,0.09,3.00


In [11]:
# ============================================================
# STRESS TESTING
# Base / Moderate Stress / Severe Stress
# ============================================================

con.execute("""
DROP TABLE IF EXISTS stress_test;
""")

con.execute("""
CREATE TABLE stress_test AS

WITH latest AS (

    SELECT *

    FROM (
        SELECT
            *,
            ROW_NUMBER() OVER (
                PARTITION BY company
                ORDER BY fiscal_year DESC
            ) AS rn
        FROM credit_metrics
    )

    WHERE rn = 1
),

scenarios AS (

    SELECT
        'Base' AS scenario,
        0.00 AS revenue_shock,
        0.00 AS margin_shock,
        1.00 AS interest_multiplier

    UNION ALL

    SELECT
        'Moderate Stress',
        -0.05,
        -0.01,
        1.10

    UNION ALL

    SELECT
        'Severe Stress',
        -0.10,
        -0.02,
        1.20
),

model AS (

    SELECT
        l.company,
        l.fiscal_year,
        s.scenario,

        l.revenue AS base_revenue,
        l.operating_margin AS base_operating_margin,
        l.operating_cash_flow AS base_ocf,
        l.capex AS base_capex,
        l.total_debt,
        l.interest_expense AS base_interest_expense,
        l.depreciation_amortization,

        s.revenue_shock,
        s.margin_shock,
        s.interest_multiplier,

        -- Revenue after scenario shock
        l.revenue * (1 + s.revenue_shock)
            AS stressed_revenue,

        -- Operating margin after scenario shock
        GREATEST(
            l.operating_margin + s.margin_shock,
            0
        ) AS stressed_operating_margin,

        -- Interest expense after stress
        l.interest_expense
            * s.interest_multiplier
            AS stressed_interest_expense,

        -- Historical cash conversion relationship
        l.operating_cash_flow
            / NULLIF(l.ebitda, 0)
            AS ocf_to_ebitda_conversion

    FROM latest l

    CROSS JOIN scenarios s
),

calculations AS (

    SELECT
        *,

        stressed_revenue
            * stressed_operating_margin
            AS stressed_ebit,

        stressed_revenue
            * stressed_operating_margin
            + depreciation_amortization
            AS stressed_ebitda

    FROM model
),

cash_flow AS (

    SELECT
        *,

        stressed_ebitda
            * ocf_to_ebitda_conversion
            AS stressed_operating_cash_flow

    FROM calculations
)

SELECT
    company,
    scenario,

    stressed_revenue,
    stressed_operating_margin,

    stressed_ebit,
    stressed_ebitda,

    stressed_interest_expense,
    stressed_operating_cash_flow,

    stressed_operating_cash_flow
        - base_capex
        AS stressed_fcf,

    stressed_ebit
        / NULLIF(stressed_interest_expense, 0)
        AS stressed_interest_coverage,

    total_debt
        / NULLIF(stressed_ebitda, 0)
        AS stressed_debt_to_ebitda,

    (
        stressed_operating_cash_flow
        - base_capex
    )
        / NULLIF(total_debt, 0)
        AS stressed_fcf_to_debt

FROM cash_flow;
""")

In [12]:
# ============================================================
# VIEW STRESS TEST RESULTS
# ============================================================

stress_results = con.execute("""
SELECT
    company,
    scenario,

    ROUND(stressed_revenue / 1000000000, 2)
        AS revenue_billions,

    ROUND(stressed_operating_margin * 100, 2)
        AS operating_margin_pct,

    ROUND(stressed_ebitda / 1000000000, 2)
        AS ebitda_billions,

    ROUND(stressed_fcf / 1000000000, 2)
        AS fcf_billions,

    ROUND(stressed_interest_coverage, 2)
        AS interest_coverage,

    ROUND(stressed_debt_to_ebitda, 2)
        AS debt_to_ebitda,

    ROUND(stressed_fcf_to_debt, 2)
        AS fcf_to_debt

FROM stress_test

ORDER BY
    company,

    CASE scenario
        WHEN 'Base' THEN 1
        WHEN 'Moderate Stress' THEN 2
        WHEN 'Severe Stress' THEN 3
    END;
""").df()

stress_results

,company,scenario,revenue_billions,operating_margin_pct,ebitda_billions,fcf_billions,interest_coverage,debt_to_ebitda,fcf_to_debt
0,Costco,Base,275.24,3.77,12.81,7.84,67.42,0.45,1.35
1,Costco,Moderate Stress,261.47,2.77,9.68,4.57,42.79,0.60,0.79
2,Costco,Severe Stress,247.71,1.77,6.82,1.60,23.76,0.85,0.28
3,Home Depot,Base,164.68,12.68,24.16,12.65,8.66,2.04,0.26
4,Home Depot,Moderate Stress,156.45,11.68,21.55,10.88,6.89,2.29,0.22
5,Home Depot,Severe Stress,148.21,10.68,19.11,9.23,5.47,2.58,0.19
6,Lowes,Base,86.29,11.77,12.09,7.65,6.83,3.29,0.19
7,Lowes,Moderate Stress,81.97,10.77,10.77,6.57,5.40,3.70,0.16
8,Lowes,Severe Stress,77.66,9.77,9.53,5.56,4.25,4.18,0.14
9,Target,Base,104.78,4.88,7.73,2.84,11.50,1.86,0.20


In [13]:
# ============================================================
# STRESS RESILIENCE SCORE
# ============================================================

con.execute("""
DROP TABLE IF EXISTS stress_scores;
""")

con.execute("""
CREATE TABLE stress_scores AS

WITH severe AS (

    SELECT *
    FROM stress_test
    WHERE scenario = 'Severe Stress'

),

scores AS (

    SELECT
        company,

        stressed_interest_coverage,
        stressed_debt_to_ebitda,
        stressed_fcf_to_debt,
        stressed_fcf,

        -- Interest coverage under severe stress
        CASE
            WHEN stressed_interest_coverage >= 8 THEN 5
            WHEN stressed_interest_coverage >= 5 THEN 4
            WHEN stressed_interest_coverage >= 3 THEN 3
            WHEN stressed_interest_coverage >= 1.5 THEN 2
            ELSE 1
        END AS stress_coverage_score,

        -- Leverage under severe stress
        CASE
            WHEN stressed_debt_to_ebitda <= 1 THEN 5
            WHEN stressed_debt_to_ebitda <= 2 THEN 4
            WHEN stressed_debt_to_ebitda <= 3 THEN 3
            WHEN stressed_debt_to_ebitda <= 4 THEN 2
            ELSE 1
        END AS stress_leverage_score,

        -- FCF available for debt repayment
        CASE
            WHEN stressed_fcf_to_debt >= 0.50 THEN 5
            WHEN stressed_fcf_to_debt >= 0.30 THEN 4
            WHEN stressed_fcf_to_debt >= 0.20 THEN 3
            WHEN stressed_fcf_to_debt >= 0.10 THEN 2
            ELSE 1
        END AS stress_fcf_score

    FROM severe
)

SELECT
    *,

    ROUND(
          stress_coverage_score * 0.40
        + stress_leverage_score * 0.35
        + stress_fcf_score * 0.25
    , 2) AS stress_score

FROM scores;
""")

In [14]:
# ============================================================
# FINAL CREDIT SCORE + RANKING
# ============================================================

final_scores = con.execute("""
SELECT
    p.company,

    p.pre_stress_score,
    s.stress_score,

    ROUND(
        p.pre_stress_score * 0.85
        + s.stress_score * 0.15
    , 2) AS final_credit_score,

    CASE
        WHEN (
            p.pre_stress_score * 0.85
            + s.stress_score * 0.15
        ) >= 4.25
            THEN 'Strong'

        WHEN (
            p.pre_stress_score * 0.85
            + s.stress_score * 0.15
        ) >= 3.50
            THEN 'Moderate / Strong'

        WHEN (
            p.pre_stress_score * 0.85
            + s.stress_score * 0.15
        ) >= 2.75
            THEN 'Moderate'

        WHEN (
            p.pre_stress_score * 0.85
            + s.stress_score * 0.15
        ) >= 2.00
            THEN 'Weak / Moderate'

        ELSE 'Weak'

    END AS credit_rating

FROM preliminary_credit_scores p

JOIN stress_scores s
    ON p.company = s.company

ORDER BY final_credit_score DESC;
""").df()

final_scores

,company,pre_stress_score,stress_score,final_credit_score,credit_rating
0,Costco,4.30,4.50,4.33,Strong
1,Walmart,3.90,3.25,3.80,Moderate / Strong
2,Home Depot,3.65,3.15,3.58,Moderate / Strong
3,Target,3.35,2.90,3.28,Moderate
4,Lowes,3.00,2.05,2.86,Moderate


In [15]:
# ============================================================
# BUILD EXCEL CREDIT MODEL
# ============================================================

import pandas as pd
from pathlib import Path
from openpyxl import load_workbook
from openpyxl.styles import Font
from openpyxl.utils import get_column_letter


# ============================================================
# FILE LOCATION
# ============================================================

EXCEL_DIR = PROJECT_ROOT / "excel"
EXCEL_DIR.mkdir(parents=True, exist_ok=True)

excel_path = EXCEL_DIR / "credit_model.xlsx"


# ============================================================
# GET LATEST FINANCIAL DATA FOR SCENARIO MODEL
# ============================================================

latest_base = con.execute("""
WITH ranked AS (

    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY company
            ORDER BY fiscal_year DESC
        ) AS rn

    FROM credit_metrics
)

SELECT
    company,
    fiscal_year,
    revenue,
    operating_margin,
    operating_cash_flow,
    capex,
    total_debt,
    interest_expense,
    depreciation_amortization,
    ebitda,

    operating_cash_flow
        / NULLIF(ebitda, 0)
        AS ocf_to_ebitda_conversion

FROM ranked

WHERE rn = 1

ORDER BY company;
""").df()


# ============================================================
# WRITE MAIN DATA SHEETS
# ============================================================

with pd.ExcelWriter(
    excel_path,
    engine="openpyxl"
) as writer:

    financials.to_excel(
        writer,
        sheet_name="Historical Financials",
        index=False
    )

    metrics.to_excel(
        writer,
        sheet_name="Credit Metrics",
        index=False
    )

    final_scores.to_excel(
        writer,
        sheet_name="Credit Scores",
        index=False
    )


# ============================================================
# OPEN WORKBOOK
# ============================================================

wb = load_workbook(excel_path)

ws = wb.create_sheet("Scenario Analysis")


# ============================================================
# SCENARIO ASSUMPTIONS
# ============================================================

ws["A1"] = "Scenario"
ws["B1"] = "Revenue Shock"
ws["C1"] = "Margin Shock"
ws["D1"] = "Interest Multiplier"

scenario_assumptions = [
    ["Base", 0.00, 0.00, 1.00],
    ["Moderate Stress", -0.05, -0.01, 1.10],
    ["Severe Stress", -0.10, -0.02, 1.20]
]

for row_num, scenario in enumerate(
    scenario_assumptions,
    start=2
):
    for col_num, value in enumerate(
        scenario,
        start=1
    ):
        ws.cell(
            row=row_num,
            column=col_num,
            value=value
        )


# ============================================================
# SCENARIO MODEL HEADERS
# ============================================================

headers = [
    "Company",
    "Scenario",
    "Base Revenue",
    "Base Operating Margin",
    "Base OCF",
    "CapEx",
    "Total Debt",
    "Base Interest Expense",
    "D&A",
    "OCF / EBITDA Conversion",
    "Revenue Shock",
    "Margin Shock",
    "Interest Multiplier",
    "Stressed Revenue",
    "Stressed Operating Margin",
    "Stressed EBIT",
    "Stressed EBITDA",
    "Stressed Interest Expense",
    "Stressed OCF",
    "Stressed FCF",
    "Interest Coverage",
    "Debt / EBITDA",
    "FCF / Debt"
]

header_row = 7

for col_num, header in enumerate(
    headers,
    start=1
):
    cell = ws.cell(
        row=header_row,
        column=col_num,
        value=header
    )

    cell.font = Font(bold=True)


# ============================================================
# BUILD 15 SCENARIO ROWS
# 5 COMPANIES × 3 SCENARIOS
# ============================================================

scenarios = [
    "Base",
    "Moderate Stress",
    "Severe Stress"
]

row_num = 8

for company_row in latest_base.itertuples():

    for scenario in scenarios:

        ws.cell(
            row=row_num,
            column=1,
            value=company_row.company
        )

        ws.cell(
            row=row_num,
            column=2,
            value=scenario
        )

        ws.cell(
            row=row_num,
            column=3,
            value=company_row.revenue
        )

        ws.cell(
            row=row_num,
            column=4,
            value=company_row.operating_margin
        )

        ws.cell(
            row=row_num,
            column=5,
            value=company_row.operating_cash_flow
        )

        ws.cell(
            row=row_num,
            column=6,
            value=company_row.capex
        )

        ws.cell(
            row=row_num,
            column=7,
            value=company_row.total_debt
        )

        ws.cell(
            row=row_num,
            column=8,
            value=company_row.interest_expense
        )

        ws.cell(
            row=row_num,
            column=9,
            value=company_row.depreciation_amortization
        )

        ws.cell(
            row=row_num,
            column=10,
            value=company_row.ocf_to_ebitda_conversion
        )


        # Scenario assumptions pulled from table above
        ws.cell(
            row=row_num,
            column=11,
            value=f'=VLOOKUP(B{row_num},$A$2:$D$4,2,FALSE)'
        )

        ws.cell(
            row=row_num,
            column=12,
            value=f'=VLOOKUP(B{row_num},$A$2:$D$4,3,FALSE)'
        )

        ws.cell(
            row=row_num,
            column=13,
            value=f'=VLOOKUP(B{row_num},$A$2:$D$4,4,FALSE)'
        )


        # ====================================================
        # EXCEL FORMULAS
        # ====================================================

        # Stressed Revenue
        ws.cell(
            row=row_num,
            column=14,
            value=f'=C{row_num}*(1+K{row_num})'
        )

        # Stressed Margin
        ws.cell(
            row=row_num,
            column=15,
            value=f'=MAX(D{row_num}+L{row_num},0)'
        )

        # Stressed EBIT
        ws.cell(
            row=row_num,
            column=16,
            value=f'=N{row_num}*O{row_num}'
        )

        # Stressed EBITDA
        ws.cell(
            row=row_num,
            column=17,
            value=f'=P{row_num}+I{row_num}'
        )

        # Stressed Interest Expense
        ws.cell(
            row=row_num,
            column=18,
            value=f'=H{row_num}*M{row_num}'
        )

        # Stressed Operating Cash Flow
        ws.cell(
            row=row_num,
            column=19,
            value=f'=Q{row_num}*J{row_num}'
        )

        # Stressed Free Cash Flow
        ws.cell(
            row=row_num,
            column=20,
            value=f'=S{row_num}-F{row_num}'
        )

        # Interest Coverage
        ws.cell(
            row=row_num,
            column=21,
            value=f'=IFERROR(P{row_num}/R{row_num},0)'
        )

        # Debt / EBITDA
        ws.cell(
            row=row_num,
            column=22,
            value=f'=IFERROR(G{row_num}/Q{row_num},0)'
        )

        # FCF / Debt
        ws.cell(
            row=row_num,
            column=23,
            value=f'=IFERROR(T{row_num}/G{row_num},0)'
        )

        row_num += 1


# ============================================================
# FORMATTING
# ============================================================

for sheet in wb.worksheets:

    sheet.freeze_panes = "A2"

    for cell in sheet[1]:
        cell.font = Font(bold=True)

    for column_cells in sheet.columns:

        max_length = 0

        column_letter = get_column_letter(
            column_cells[0].column
        )

        for cell in column_cells:

            try:
                max_length = max(
                    max_length,
                    len(str(cell.value))
                )
            except:
                pass

        sheet.column_dimensions[
            column_letter
        ].width = min(
            max_length + 2,
            30
        )


# Scenario percentage formatting
for row in range(2, 5):

    ws[f"B{row}"].number_format = "0.0%"
    ws[f"C{row}"].number_format = "0.0%"


for row in range(8, 23):

    # Percentages
    ws[f"D{row}"].number_format = "0.00%"
    ws[f"K{row}"].number_format = "0.00%"
    ws[f"L{row}"].number_format = "0.00%"
    ws[f"O{row}"].number_format = "0.00%"

    # Ratios
    ws[f"J{row}"].number_format = "0.00x"
    ws[f"U{row}"].number_format = "0.00x"
    ws[f"V{row}"].number_format = "0.00x"
    ws[f"W{row}"].number_format = "0.00x"

    # Dollar values displayed in billions
    for col in [
        "C", "E", "F", "G", "H", "I",
        "N", "P", "Q", "R", "S", "T"
    ]:
        ws[f"{col}{row}"].number_format = '$0.00,,,"B"'


# Freeze scenario model header
ws.freeze_panes = "A8"


# ============================================================
# SAVE WORKBOOK
# ============================================================

wb.save(excel_path)

print("✅ Excel credit model created")
print(f"✅ Saved to: {excel_path}")

✅ Excel credit model created
✅ Saved to: ..\excel\credit_model.xlsx


In [17]:
# ============================================================
# CLEAN EXCEL FORMATTING
# ============================================================

from openpyxl import load_workbook
from openpyxl.styles import Font, Alignment
from openpyxl.utils import get_column_letter

wb = load_workbook(excel_path)

# ------------------------------------------------------------
# HISTORICAL FINANCIALS
# ------------------------------------------------------------

ws = wb["Historical Financials"]

ws.freeze_panes = "A2"
ws.auto_filter.ref = ws.dimensions

for cell in ws[1]:
    cell.font = Font(bold=True)
    cell.alignment = Alignment(horizontal="center")

# Financial columns C:O shown in $ billions
for row in range(2, ws.max_row + 1):

    for col in range(3, 16):
        ws.cell(row=row, column=col).number_format = '$0.00,,,"B"'

# Fiscal year
for row in range(2, ws.max_row + 1):
    ws.cell(row=row, column=2).number_format = "0"

# Column widths
widths = {
    "A": 16,
    "B": 12
}

for col in range(3, 16):
    widths[get_column_letter(col)] = 20

for column, width in widths.items():
    ws.column_dimensions[column].width = width


# ------------------------------------------------------------
# CREDIT SCORES
# ------------------------------------------------------------

ws_scores = wb["Credit Scores"]

ws_scores.freeze_panes = "A2"
ws_scores.auto_filter.ref = ws_scores.dimensions

for cell in ws_scores[1]:
    cell.font = Font(bold=True)

for row in range(2, ws_scores.max_row + 1):

    for col in [2, 3, 4]:
        ws_scores.cell(
            row=row,
            column=col
        ).number_format = "0.00"


# ------------------------------------------------------------
# SAVE
# ------------------------------------------------------------

wb.save(excel_path)

print("✅ Excel formatting complete")
print(excel_path)

✅ Excel formatting complete
..\excel\credit_model.xlsx


In [18]:
# ============================================================
# EXPORT DASHBOARD-READY DATASETS
# ============================================================

DASHBOARD_DIR = PROJECT_ROOT / "dashboard" / "data"
DASHBOARD_DIR.mkdir(parents=True, exist_ok=True)


# 1. Historical financials
dashboard_financials = financials.copy()


# 2. Full credit metrics history
dashboard_metrics = con.execute("""
SELECT *
FROM credit_metrics
ORDER BY company, fiscal_year;
""").df()


# 3. Latest credit metrics
dashboard_latest = con.execute("""
WITH ranked AS (
    SELECT *,
        ROW_NUMBER() OVER (
            PARTITION BY company
            ORDER BY fiscal_year DESC
        ) AS rn
    FROM credit_metrics
)

SELECT
    company,
    fiscal_year,
    revenue_growth,
    operating_margin,
    free_cash_flow,
    current_ratio,
    equity,
    total_debt,
    interest_coverage,
    ebitda,
    debt_to_ebitda,
    ocf_to_debt

FROM ranked
WHERE rn = 1
ORDER BY company;
""").df()


# 4. Stress-test results
dashboard_stress = con.execute("""
SELECT *
FROM stress_test
ORDER BY
    company,
    CASE scenario
        WHEN 'Base' THEN 1
        WHEN 'Moderate Stress' THEN 2
        WHEN 'Severe Stress' THEN 3
    END;
""").df()


# 5. Final credit scores
dashboard_scores = final_scores.copy()


# Make company name presentation-ready
datasets = [
    dashboard_financials,
    dashboard_metrics,
    dashboard_latest,
    dashboard_stress,
    dashboard_scores
]

for df in datasets:
    df["company"] = df["company"].replace(
        {"Lowes": "Lowe's"}
    )


# ============================================================
# SAVE FILES
# ============================================================

dashboard_financials.to_csv(
    DASHBOARD_DIR / "historical_financials.csv",
    index=False
)

dashboard_metrics.to_csv(
    DASHBOARD_DIR / "credit_metrics.csv",
    index=False
)

dashboard_latest.to_csv(
    DASHBOARD_DIR / "latest_credit_metrics.csv",
    index=False
)

dashboard_stress.to_csv(
    DASHBOARD_DIR / "stress_test.csv",
    index=False
)

dashboard_scores.to_csv(
    DASHBOARD_DIR / "credit_scores.csv",
    index=False
)


print("✅ Dashboard datasets created")

for file in DASHBOARD_DIR.glob("*.csv"):
    print(file.name)

✅ Dashboard datasets created
credit_metrics.csv
credit_scores.csv
historical_financials.csv
latest_credit_metrics.csv
stress_test.csv
